In [ ]:
# Install dependencies
%pip install google-genai python-dotenv

In [ ]:
# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Create Gemini client
import os
import logging
from google import genai

# Suppress informational SDK notice about Automatic Function Calling (AFC)
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
model = "gemini-3.5-flash-lite"

In [ ]:
# Make a Single-Turn Request
# NOTE: In Gemini, token limits are OPTIONAL and passed in `config` as `max_output_tokens`.
response = client.models.generate_content(
    model=model,
    contents="What is an AI engineer?",
    config={
        "max_output_tokens": 1000  # Equivalent to Claude's max_tokens (optional in Gemini)
    }
)

In [ ]:
# View response text
print(response.text)

## Multi-turn Conversations (Gemini Counterpart)

### Statelessness in Gemini
Just like Claude, Google Gemini is completely **stateless**. The API does not retain past queries or responses on Google's servers.

To build a conversational experience:
1. Maintain a list of conversation turns in your code.
2. Pass the full history with every follow-up call.

### Key Syntax Difference Between Claude and Gemini:
- **Anthropic Claude**: Role is `"assistant"`, message dictionary is `{"role": "...", "content": text}`.
- **Google Gemini**: Role is `"model"`, message dictionary is `{"role": "...", "parts": [{"text": text}]}`.

In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user", "parts": [{"text": text}]}
    messages.append(user_message)

def add_model_message(messages, text):
    model_message = {"role": "model", "parts": [{"text": text}]}
    messages.append(model_message)

def chat(messages):
    response = client.models.generate_content(
        model=model,
        contents=messages,
        config={"max_output_tokens": 1000},
    )
    return response.text

In [ ]:
# Make a starting list of messages
messages = []

# Add in the initial user question of "Define quantum computing in one sentence"
add_user_message(messages, "Define quantum computing in one sentence")

# Pass the list of messages into 'chat' to get an answer
answer = chat(messages)
print("Turn 1 Answer:", answer.strip())

# Take the answer and add it as a model message into our list
add_model_message(messages, answer)

# Add in the user's follow-up question
add_user_message(messages, "Write another sentence")

# Call chat again with the list of messages to get a final answer
answer = chat(messages)
print("\nTurn 2 Answer:", answer.strip())

### Bonus: Gemini's Built-In Chat Session
While the manual message history pattern above is universal across all LLM APIs, the `google-genai` SDK also provides a convenient built-in chat session (`client.chats.create()`) that manages message history automatically:

In [ ]:
# Built-in Gemini chat session
chat_session = client.chats.create(model=model)

response1 = chat_session.send_message("Define quantum computing in one sentence")
print("Turn 1:", response1.text.strip())

response2 = chat_session.send_message("Write another sentence")
print("\nTurn 2:", response2.text.strip())